<a href="https://colab.research.google.com/github/Mmbsaksd/transformers/blob/main/Annotated_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Prelims**

In [1]:
# # Uncomment for colab
# #
!pip install -q  GPUtil
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 91.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torch.utils.data import DataLoader, Dataset
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from collections import Counter
from datasets import load_dataset

warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

In [3]:
def is_interactive_notebook():
    return __name__ == "__main__"


def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)


def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)


class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr": 0}]
        None

    def step(self):
        None

    def zero_grad(self, set_to_none=False):
        None


class DummyScheduler:
    def step(self):
        None

### Background

Earlier models such as Extended Neural GPU, ByteNet, and ConvS2S also tried to reduce step-by-step computation. They used convolutional neural networks (CNNs) to process many positions in the input at the same time.

However, these models had a problem when two words were far apart in a sentence. The farther apart the words were, the more computation was needed to connect them. ConvS2S needed more computation as the distance increased, while ByteNet increased more slowly but still needed additional computation.

The Transformer solves this problem using **self-attention**. Self-attention allows any word to directly look at other words in the sequence, even when they are far apart. This makes it easier for the model to learn relationships between distant words.

Self-attention was already used in some NLP tasks before the Transformer. However, the Transformer was the first model to use self-attention as the main mechanism for processing both the input and output, without using RNNs or CNNs.

In simple terms:

**Older models:** Farther words → more computation to connect them.

**Transformer:** Farther words → can still directly connect through self-attention.


## **Part 1: Model Architecture**

## Model Architecture

Most sequence-to-sequence models use two main parts: an **Encoder** and a **Decoder**.

The **Encoder** takes the input sequence and converts it into useful numerical representations. For example, the input words are converted into vectors, and the Encoder uses the relationships between the words to create better representations.

The **Decoder** uses these representations to generate the output sequence. It generates the output **one token at a time**. When generating the next token, it can use the tokens it has already generated.

The Transformer follows this Encoder-Decoder structure, but instead of using RNNs or CNNs to process the sequence, it mainly uses **attention mechanisms**.

The Encoder is made up of several identical layers. Each layer contains two main parts:

1. **Multi-Head Self-Attention** – allows each word to look at other words in the input and understand their relationships.
2. **Feed-Forward Network** – processes the information from the attention layer further.

The Decoder is also made up of several identical layers. Each layer contains three main parts:

1. **Masked Multi-Head Self-Attention** – allows the decoder to look at previously generated words, but prevents it from looking at future words.
2. **Encoder-Decoder Attention** – allows the decoder to look at the information produced by the Encoder.
3. **Feed-Forward Network** – processes the information further.

The Transformer also uses **residual connections and normalization** around these parts to help the network train effectively.

In simple terms:

**Input → Encoder → Information → Decoder → Output**

The main idea is that the **Encoder understands the input**, and the **Decoder uses that information to generate the output one token at a time**.


In [ ]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)
